# Prefilter Training Dataset: flat `.npy` arrays

Creating a training dataset for the **prefilter** neutrino classification model.

### Vision

**Data layout**: flat `.npy` arrays (memory-mappable), one file per field:
- `features.npy` — `(total_hits, 5)` float32, all events packed contiguously
- `offsets.npy` — `(n_events+1,)` int64, event boundaries into features
- `labels.npy` — `(n_events,)` float32, soft labels
- `signal_hits.npy` — `(n_events,)` int32, number of signal hits per event
- `signal_strings.npy` — `(n_events,)` int32, number of unique signal strings per event
- `particle_types.npy` — `(n_events,)` int8, encoded particle type (0=muatm, 1=nuatm, 2=nue2)

**Pipeline**:
1. Select HDF5 parts (from Grisha's train split + extras for class balance)
2. Read HDF5 **sequentially** (part-by-part) — fast, no random access
3. Compute per-event: `n_signal_hits`, `n_unique_signal_strings`, soft labels
4. Rebalance classes (signal vs background, neutrino subtypes)
5. Shuffle events
6. Write flat `.npy` arrays

**Why flat `.npy`?**
- `np.load(mmap_mode="r")` → zero-copy memory mapping, zero RAM overhead
- Single contiguous array → `features[offsets[i]:offsets[i+1]]` is one slice, microseconds
- No HDF5 tree traversal, no decompression, no per-event I/O overhead
- Rebalancing and shuffling are baked in — DataLoader can use `shuffle=False` or just shuffle offsets

### Source data
- `/net/62/home3/ivkhar/Baikal/data/h5s/baikal_mc_merged.h5` (890 GB, gzip-compressed)

### Output directory
- `/home2/albert/Baikal/data/prefilter_train_npy/`

In [1]:
import os
import logging
import time
from pathlib import Path
from typing import List, Optional

import polars as pl
import h5py
import numpy as np

In [4]:
DEFAULT_H5_PATH = Path("../data/h5datasets/baikal_mc_merged.h5")
SOURCE_CATALOG = Path("../h5_catalogs/catalogs_mc_signoise_normed/train.parquet")

MUATM_CATALOG = Path("../h5_catalogs/catalogs_mc_merged/muatm_2020.parquet")
NUATM_CATALOG = Path("../h5_catalogs/catalogs_mc_merged/nuatm_2020.parquet")
NUE2_CATALOG = Path("../h5_catalogs/catalogs_mc_merged/nue2_2020.parquet")

## Selecting parts from `...merged_mc.h5` to store

### Load parts, that to be included obligatory

In [5]:
# Lazy column loading
norm_train_ids = pl.scan_parquet(SOURCE_CATALOG).select("event_id").collect()
df_temp = norm_train_ids.with_columns(
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.first().alias("particle_type"),
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.get(1).cast(pl.Int32).alias("part_num"),
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.get(2).cast(pl.Int32).alias("part_event_id"),
)

parts_to_store = {
    'muatm': [],
    'nuatm': [],
    'nue2': []
}
parts_counts_to_store = {
    'muatm': None,
    'nuatm': None,
    'nue2': None
}
for ptype in parts_to_store:
    part_ids_with_counts = df_temp.filter(pl.col('particle_type')==ptype)['part_num'].value_counts().sort('part_num')
    parts_to_store[ptype] = part_ids_with_counts['part_num'].to_list()
    parts_counts_to_store[ptype] = part_ids_with_counts['count']
    print(f"For {ptype}:")
    print(f"\tnumber of parts: {part_ids_with_counts['count'].shape[0]}")
    print(f"\tnumber of events: {part_ids_with_counts['count'].sum()}")
    print(f"\tmean number of events: {part_ids_with_counts['count'].mean()}")

For muatm:
	number of parts: 374
	number of events: 10890482
	mean number of events: 29118.935828877005
For nuatm:
	number of parts: 187
	number of events: 6289473
	mean number of events: 33633.545454545456
For nue2:
	number of parts: 88
	number of events: 1664273
	mean number of events: 18912.19318181818


### Get numbers of parts to add more

In [4]:
nuatm_already_number = parts_counts_to_store['nuatm'].sum()
muatm_already_number = parts_counts_to_store['muatm'].sum()
nue2_already_number = parts_counts_to_store['nue2'].sum()
assert nuatm_already_number*2 > muatm_already_number
assert nuatm_already_number > nue2_already_number

mean_nuatm_per_part = parts_counts_to_store['nuatm'].mean()
mean_muatm_per_part = parts_counts_to_store['muatm'].mean()
mean_nue2_per_part = parts_counts_to_store['nue2'].mean()

NUM_PARTS_TO_ADD = {
    'muatm': int(np.round((nuatm_already_number*2 - muatm_already_number)/mean_muatm_per_part)),
    'nuatm': int(np.round((nuatm_already_number-nuatm_already_number)/mean_nuatm_per_part)),
    'nue2': int(np.round((nuatm_already_number - nue2_already_number)/mean_nue2_per_part)),
}
NUM_PARTS_TO_ADD

{'muatm': 58, 'nuatm': 0, 'nue2': 245}

### Randomly select parts to add from main catalogs

In [5]:
norm_train_ids = pl.scan_parquet(SOURCE_CATALOG).select("event_id").collect()
df_temp = norm_train_ids.with_columns(
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.first().alias("particle_type"),
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.get(1).cast(pl.Int32).alias("part_num"),
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.get(2).cast(pl.Int32).alias("part_event_id"),
)

In [6]:
mu_parts_already: list = parts_to_store['muatm']
new_mu_df = pl.scan_parquet(MUATM_CATALOG).select('h5_part_num').group_by('h5_part_num').len().filter(~pl.col('h5_part_num').is_in(mu_parts_already)).collect()
new_mu_parts: list = new_mu_df['h5_part_num'].sample(fraction=1, shuffle=True)[:NUM_PARTS_TO_ADD['muatm']].to_list()

print(f"Required {nuatm_already_number*2 - muatm_already_number:,} events.")
print(f"Will collect: {new_mu_df.filter(pl.col('h5_part_num').is_in(new_mu_parts))['len'].sum():,}")

Required 1,688,464 events.
Will collect: 1,658,213


In [7]:
nuatm_parts_already: list = parts_to_store['nuatm']
new_nuatm_df = pl.scan_parquet(NUATM_CATALOG).select('h5_part_num').group_by('h5_part_num').len().filter(~pl.col('h5_part_num').is_in(nuatm_parts_already)).collect()
new_nuatm_parts: list = new_nuatm_df['h5_part_num'].sample(fraction=1, shuffle=True)[:NUM_PARTS_TO_ADD['nuatm']].to_list()

print(f"Required {0:,} events.")
print(f"Will collect: {new_nuatm_df.filter(pl.col('h5_part_num').is_in(new_nuatm_parts))['len'].sum():,}")

Required 0 events.
Will collect: 0


In [8]:
nue2_parts_already: list = parts_to_store['nue2']
new_nue2_df = pl.scan_parquet(NUE2_CATALOG).select('h5_part_num').group_by('h5_part_num').len().filter(~pl.col('h5_part_num').is_in(nue2_parts_already)).collect()
new_nue2_parts: list = new_nue2_df['h5_part_num'].sample(fraction=1, shuffle=True)[:NUM_PARTS_TO_ADD['nue2']].to_list()

print(f"Required {nuatm_already_number - nue2_already_number:,} events.")
print(f"Will collect: {new_nue2_df.filter(pl.col('h5_part_num').is_in(new_nue2_parts))['len'].sum():,}")

Required 4,625,200 events.
Will collect: 4,709,053


In [9]:
parts_to_store_final = {}
parts_to_store_final['muatm'] = parts_to_store['muatm'] + new_mu_parts
parts_to_store_final['nuatm'] = parts_to_store['nuatm'] + new_nuatm_parts
parts_to_store_final['nue2'] = parts_to_store['nue2'] + new_nue2_parts

In [14]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

import json
json_path = OUTPUT_DIR / "prefilter_h5_parts.json"
with open(json_path, "w") as f:
    json.dump(parts_to_store_final, f)

In [15]:
json_path

PosixPath('/home2/albert/Baikal/datasets/baikal_mc2020_prefilter/prefilter_h5_parts.json')